# Ngày 6 — Gọi OpenAI API lần đầu

Mục tiêu: gọi được API, hiểu request/response, quan sát token ↔ chi phí ↔ độ trễ.

**Chuẩn bị:**
- `pip install openai python-dotenv`
- Đặt key vào biến môi trường: `setx OPENAI_API_KEY "sk-..."` (Windows) hoặc file `.env` (đã thêm `.env` vào `.gitignore`).
- **Tuyệt đối không viết key thẳng vào notebook.**

In [ ]:
import os
import time
from openai import OpenAI

# from dotenv import load_dotenv; load_dotenv()  # bỏ comment nếu dùng file .env

assert os.environ.get("OPENAI_API_KEY"), "Chưa đặt OPENAI_API_KEY trong env!"
client = OpenAI()  # tự đọc OPENAI_API_KEY từ env
MODEL = "gpt-4o-mini"
print("Sẵn sàng gọi model:", MODEL)

## 1. Hàm gọi API — trả về nội dung + token + độ trễ

In [ ]:
def ask(prompt, temperature=0.7, max_tokens=200, system="Bạn là trợ lý hữu ích, trả lời ngắn gọn."):
    """Gọi OpenAI, trả về dict {content, prompt_tokens, completion_tokens, latency_s}."""
    t0 = time.perf_counter()
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": prompt},
            ],
            temperature=temperature,
            max_tokens=max_tokens,
        )
    except Exception as e:  # openai.APIError, RateLimitError, ...
        return {"content": f"[LỖI] {type(e).__name__}: {e}", "prompt_tokens": 0, "completion_tokens": 0, "latency_s": None}
    dt = time.perf_counter() - t0
    return {
        "content": resp.choices[0].message.content,
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "latency_s": round(dt, 2),
    }

ask("Chào bạn, giới thiệu bản thân trong 1 câu.")

## 2. Chạy tác vụ trên ≥3 input

Chọn 1 tác vụ: **tóm tắt** hoặc **phân loại cảm xúc**. Ví dụ dưới là phân loại cảm xúc.

In [ ]:
inputs = [
    "Sản phẩm giao nhanh, đóng gói cẩn thận, rất hài lòng!",
    "Chờ mãi không thấy hàng, gọi hỗ trợ thì không ai nghe máy.",
    "Hàng ổn, đúng mô tả, không có gì đặc biệt.",
]

for text in inputs:
    # TODO: đổi prompt cho hợp tác vụ bạn chọn
    r = ask(f"Phân loại cảm xúc (Tích cực/Tiêu cực/Trung tính), chỉ trả 1 từ:\n\"{text}\"", temperature=0, max_tokens=5)
    print(f"- {text}\n  => {r['content']}  (tokens in/out: {r['prompt_tokens']}/{r['completion_tokens']}, {r['latency_s']}s)\n")

## 3. Thí nghiệm: đổi `temperature` và `max_tokens`

Cùng 1 prompt, quan sát khác biệt về nội dung, token, độ trễ.

In [ ]:
prompt = "Viết một khẩu hiệu quảng cáo cho quán cà phê."

for temp in [0, 1]:
    r = ask(prompt, temperature=temp, max_tokens=40)
    print(f"temperature={temp}: {r['content']}  ({r['latency_s']}s)")

# TODO: thử thêm max_tokens nhỏ (VD 10) vs lớn (VD 200) và ghi lại khác biệt độ dài + token

## 4. Ước tính chi phí

> Cập nhật đơn giá theo openai.com/pricing (đơn vị USD / 1 triệu token). Số dưới chỉ là ví dụ cho `gpt-4o-mini`.

In [ ]:
PRICE_IN = 0.15   # USD / 1M input tokens  (VD gpt-4o-mini — KIỂM TRA LẠI)
PRICE_OUT = 0.60  # USD / 1M output tokens (VD gpt-4o-mini — KIỂM TRA LẠI)

def cost(prompt_tokens, completion_tokens):
    return prompt_tokens / 1e6 * PRICE_IN + completion_tokens / 1e6 * PRICE_OUT

r = ask("Tóm tắt lợi ích của việc đọc sách trong 2 câu.", temperature=0, max_tokens=80)
print(r["content"])
print(f"Chi phí ước tính: ${cost(r['prompt_tokens'], r['completion_tokens']):.6f}")

## Diễn giải / nhận xét

_(Điền: temperature ảnh hưởng thế nào tới nội dung? max_tokens ảnh hưởng độ dài & chi phí? độ trễ có ổn định không?)_